# Classificação Robusta da Maturação Cervical Vertebral (CVM) em Múltiplos Equipamentos
## Classificação Direta com ResNet-50 e Análise de Robustez por Equipamento

**Dataset:** Aariz Cephalometric Dataset (~1000 imagens, 7 diferentes equipamentos de raio-X)  
**Tarefa:** Classificação multiclasse — CVM-S1 a CVM-S6  

---

### Metodologia de Análise e Segmentação

A variação de domínio (*Domain Shift*) causada por diferentes equipamentos de aquisição é um dos maiores obstáculos para a implantação clínica de modelos de IA em radiografias. Modelos treinados em imagens de um equipamento podem não generalizar para imagens de outros, pois capturam **padrões espúrios** (contraste específico, marcas d'água, artefatos de colimação) em vez da morfologia óssea relevante.

Para avaliar e mitigar esse problema, nossa abordagem consiste em:

1. **Classificação Direta (Estágio Único):** Um classificador ResNet-50 com fine-tuning recebe a radiografia cefalométrica completa, sem segmentação anatômica explícita. O modelo é forçado a aprender a morfologia das vértebras C2-C4 a partir da imagem inteira.

2. **Análise de Robustez por Equipamento:** A acurácia é estratificada por cada um dos 7 equipamentos de raio-X, permitindo detectar em quais máquinas o modelo sofre de domain shift.

3. **Leave-One-Equipment-Out (LOEO) Cross-Validation:** O modelo é treinado em 6 equipamentos e testado no 7º (held-out), repetindo para todos. Este é o teste mais rigoroso: mede a capacidade de generalização do modelo para **equipamentos nunca vistos** durante o treino.

**Hipótese Científica:** A avaliação sistemática por equipamento — tanto no split padrão quanto no LOEO — permite quantificar o grau de domain shift e identificar quais equipamentos apresentam maior viés. Modelos com baixa variância entre equipamentos são considerados robustos.

---

### Metodologia de Pré-processamento

- **CLAHE (Contrast Limited Adaptive Histogram Equalization):** Aplicado no canal L do espaço de cor LAB para melhorar o contraste local das radiografias, padronizando a iluminação entre diferentes equipamentos.
- **Aumento de Dados:** Rotações aleatórias (±5°), cortes aleatórios, flips horizontais, e ajustes de brilho/contraste para melhorar a generalização.
- **Weighted Loss:** Pesos inversamente proporcionais à frequência de cada classe para lidar com o desbalanceamento severo (CVM-S1: ~18 amostras, CVM-S5: ~311).

---
## 0. Instalação e Imports

In [ ]:
import os, json, random, shutil, copy
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms, models

from sklearn.metrics import classification_report, confusion_matrix, cohen_kappa_score

# Reprodutibilidade
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

# Dispositivo
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo em uso: {DEVICE}')
print(f'PyTorch version: {torch.__version__}')

---
## 1. Configuração dos Caminhos e Parâmetros

In [ ]:
# Caminho do dataset
ORIGINAL_ROOT = Path('Aariz_extracted/Aariz')

# Caminhos de saída
RESNET_WEIGHTS = Path('best_resnet_cvm.pth')
LOEO_DIR       = Path('resultados_loeo')

# Hiperparâmetros
IMG_SIZE     = 224
BATCH_SIZE   = 16
NUM_CLASSES  = 6
EPOCHS       = 50
LR           = 1e-4
LR_STEP      = 10
LR_GAMMA     = 0.5

# LOEO
LOEO_EPOCHS  = 20
LOEO_LR      = 5e-5

CLASS_NAMES  = ['CVM-S1', 'CVM-S2', 'CVM-S3', 'CVM-S4', 'CVM-S5', 'CVM-S6']

assert ORIGINAL_ROOT.exists(), f'Dataset nao encontrado em {ORIGINAL_ROOT}'
print('Estrutura OK:', ORIGINAL_ROOT)
LOEO_DIR.mkdir(exist_ok=True)

---
## 2. Carregamento dos Dados e Análise Exploratória

In [ ]:
def load_split(split: str) -> pd.DataFrame:
    """Carrega imagens e labels CVM de um split (train/valid/test)."""
    img_dir = ORIGINAL_ROOT / split / 'Cephalograms'
    ann_dir = ORIGINAL_ROOT / split / 'Annotations' / 'CVM Stages'

    records = []
    for json_path in sorted(ann_dir.glob('*.json')):
        with open(json_path) as f:
            data = json.load(f)

        cvm_obj = data.get('cvm_stage', {})
        stage   = cvm_obj.get('title', '').strip()

        if not stage:
            continue

        img_path = None
        for ext in ['.png', '.jpg', '.jpeg', '.bmp']:
            candidate = img_dir / (json_path.stem + ext)
            if candidate.exists():
                img_path = candidate
                break

        if img_path and stage in CLASS_NAMES:
            records.append({
                'image_path': str(img_path),
                'ceph_id':    data.get('ceph_id', json_path.stem),
                'label':      stage,
                'label_idx':  CLASS_NAMES.index(stage)
            })

    df = pd.DataFrame(records)
    print(f'[{split}] {len(df)} imagens carregadas.')
    return df


df_train = load_split('train')
df_valid = load_split('valid')
df_test  = load_split('test')

print(f'\nTotal: Treino={len(df_train)}, Validacao={len(df_valid)}, Teste={len(df_test)}')

In [ ]:
# Distribuição de classes
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
colors = sns.color_palette('viridis', NUM_CLASSES)

for ax, (df, title) in zip(axes, [
    (df_train, f'Treino ({len(df_train)})'),
    (df_valid, f'Validacao ({len(df_valid)})'),
    (df_test,  f'Teste ({len(df_test)})')
]):
    counts = df['label'].value_counts().reindex(CLASS_NAMES, fill_value=0)
    ax.bar(CLASS_NAMES, counts.values, color=colors)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('Estagio CVM')
    ax.set_ylabel('Quantidade')
    for i, v in enumerate(counts.values):
        ax.text(i, v + 1, str(v), ha='center', fontsize=10)

plt.suptitle('Distribuicao de Estagios CVM por Split', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

# Pesos por classe (inverso da frequencia)
counts_train = df_train['label'].value_counts().reindex(CLASS_NAMES, fill_value=1)
class_weights = torch.tensor(
    [1.0 / max(counts_train[c], 1) for c in CLASS_NAMES], dtype=torch.float
).to(DEVICE)
class_weights = class_weights / class_weights.sum() * NUM_CLASSES

print('\nPesos por classe (class_weight) para CrossEntropyLoss:')
for name, w in zip(CLASS_NAMES, class_weights.cpu()):
    print(f'  {name}: {w:.4f}')

---
## 3. Dataset PyTorch com CLAHE e Data Augmentation

In [ ]:
def apply_clahe(img_np: np.ndarray) -> np.ndarray:
    """CLAHE no canal L do LAB. Melhora contraste local de radiografias."""
    lab = cv2.cvtColor(img_np, cv2.COLOR_RGB2LAB)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    lab[:, :, 0] = clahe.apply(lab[:, :, 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)


class CVMDataset(Dataset):
    """Dataset para classificacao CVM a partir das imagens originais."""
    def __init__(self, df: pd.DataFrame, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        label = int(row['label_idx'])
        img_path = row['image_path']
        img_bgr = cv2.imread(img_path)

        if img_bgr is None:
            img_rgb = np.zeros((224, 224, 3), dtype=np.uint8)
        else:
            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
            img_rgb = apply_clahe(img_rgb)

        from PIL import Image
        img_pil = Image.fromarray(img_rgb)
        if self.transform:
            img_pil = self.transform(img_pil)
        return img_pil, label


IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

transform_train = transforms.Compose([
    transforms.Resize((IMG_SIZE + 20, IMG_SIZE + 20)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.RandomRotation(degrees=5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

transform_eval = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

train_ds = CVMDataset(df_train, transform=transform_train)
valid_ds = CVMDataset(df_valid, transform=transform_eval)
test_ds  = CVMDataset(df_test,  transform=transform_eval)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'\nBatches - Treino: {len(train_loader)} | Validacao: {len(valid_loader)} | Teste: {len(test_loader)}')

---
## 4. Modelo ResNet-50 com Fine-Tuning

In [ ]:
def build_resnet50(num_classes: int, freeze_strategy='partial') -> nn.Module:
    """
    Constroi ResNet-50 pre-treinada.
    freeze_strategy: 'partial' (layer2-4 descongelados) ou 'full' (tudo descongelado).
    """
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

    if freeze_strategy == 'partial':
        for param in model.parameters():
            param.requires_grad = False
        for param in model.layer2.parameters():
            param.requires_grad = True
        for param in model.layer3.parameters():
            param.requires_grad = True
        for param in model.layer4.parameters():
            param.requires_grad = True
    else:  # full
        for param in model.parameters():
            param.requires_grad = True

    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_features, num_classes)
    )
    return model


model = build_resnet50(NUM_CLASSES).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR
)

scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=LR_STEP, gamma=LR_GAMMA)

print(f'Modelo: ResNet-50 ({sum(p.numel() for p in model.parameters()):,} parametros)')
print(f'Treinaveis: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

---
## 5. Treinamento do Classificador CVM

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    pbar = tqdm(loader, desc='Train')
    for inputs, labels in pbar:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{100*correct/total:.2f}%'})
    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc  = correct / total
    return epoch_loss, epoch_acc


def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for inputs, labels in tqdm(loader, desc='Eval'):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc  = np.mean(np.array(all_preds) == np.array(all_labels))
    return epoch_loss, epoch_acc, np.array(all_preds), np.array(all_labels)


best_val_acc = 0.0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

print('\n' + '='*60)
print('INICIANDO TREINAMENTO')
print('='*60 + '\n')

for epoch in range(1, EPOCHS + 1):
    print(f'\nEpoca {epoch}/{EPOCHS}')
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_loss, val_acc, _, _ = evaluate(model, valid_loader, criterion, DEVICE)
    scheduler.step()
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    print(f'Train Loss: {train_loss:.4f} | Train Acc: {100*train_acc:.2f}%')
    print(f'Val Loss: {val_loss:.4f} | Val Acc: {100*val_acc:.2f}%')
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), str(RESNET_WEIGHTS))
        print(f'Novo melhor modelo salvo! (Val Acc: {100*val_acc:.2f}%)')

print(f'\nTreinamento concluido! Melhor val_acc: {100*best_val_acc:.2f}%')

---
## 6. Curvas de Treinamento

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history['train_loss'], label='Train Loss', linewidth=2)
axes[0].plot(history['val_loss'], label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoca')
axes[0].set_ylabel('Loss')
axes[0].set_title('Curvas de Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['train_acc'], label='Train Acc', linewidth=2)
axes[1].plot(history['val_acc'], label='Val Acc', linewidth=2)
axes[1].axhline(y=best_val_acc, color='green', linestyle='--', alpha=0.5, label=f'Best: {100*best_val_acc:.2f}%')
axes[1].set_xlabel('Epoca')
axes[1].set_ylabel('Acuracia')
axes[1].set_title('Curvas de Acuracia')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('curvas_treinamento.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Avaliacao no Test Set

In [ ]:
if RESNET_WEIGHTS.exists():
    model.load_state_dict(torch.load(str(RESNET_WEIGHTS), map_location=DEVICE))
    print(f'Modelo carregado: {RESNET_WEIGHTS}')

test_loss, test_acc, all_preds, all_labels = evaluate(model, test_loader, criterion, DEVICE)

print(f'\n{"="*50}')
print('RESULTADOS NO TEST SET')
print(f'{"="*50}')
print(f'Test Loss: {test_loss:.4f}')
print(f'Test Acc:  {100*test_acc:.2f}%')

print(f'\nClassification Report:')
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES, digits=4))

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel('Predito')
plt.ylabel('Verdadeiro')
plt.title('Matriz de Confusao - Test Set')
plt.tight_layout()
plt.savefig('matriz_confusao.png', dpi=150, bbox_inches='tight')
plt.show()

kappa = cohen_kappa_score(all_labels, all_preds, weights='quadratic')
print(f'\nCohen\'s Kappa (quadratico): {kappa:.4f}')

---
## 8. Avaliacao de Robustez: Acuracia por Equipamento

> **Hipotese:** A avaliacao estratificada por equipamento permite quantificar o
> domain shift causado pelos 7 diferentes aparelhos de raio-X. Se o modelo for
> robusto, a acuracia deve ser consistente entre os equipamentos.

In [ ]:
machines_df = pd.read_csv(ORIGINAL_ROOT / 'cephalogram_machine_mappings.csv')
print(f'Mapeamento: {len(machines_df)} registros, {len(machines_df["machine"].unique())} equipamentos')

ceph_to_machine = dict(zip(machines_df['cephalogram_id'], machines_df['machine']))
df_test['machine'] = df_test['ceph_id'].map(ceph_to_machine)

unmapped = df_test['machine'].isna().sum()
if unmapped > 0:
    print(f'{unmapped} imagens sem mapeamento (ignoradas)')

print(f'\nDistribuicao de equipamentos no test set:')
print(df_test['machine'].value_counts())

In [ ]:
df_test_eval = df_test.dropna(subset=['machine']).copy()
print(f'Avaliando {len(df_test_eval)}/{len(df_test)} imagens com mapeamento...')

df_test_eval['pred_idx'] = [all_preds[i] for i in df_test_eval.index]
df_test_eval['correct']  = df_test_eval['label_idx'] == df_test_eval['pred_idx']

acc_by_machine = df_test_eval.groupby('machine').agg(
    accuracy=('correct', 'mean'),
    count=('correct', 'count')
).reset_index()
acc_by_machine['accuracy'] = acc_by_machine['accuracy'] * 100

print(f'\n{"="*60}')
print('Acuracia por Equipamento (Test Set)')
print(f'{"="*60}')
for _, row in acc_by_machine.iterrows():
    bar = chr(9608) * int(row['accuracy'] / 2)
    print(f'{row["machine"]:30s} | {bar:>25s} {row["accuracy"]:5.1f}% ({int(row["count"])} amostras)')

mean_acc = acc_by_machine['accuracy'].mean()
std_acc  = acc_by_machine['accuracy'].std()

print(f'\nEstatisticas de Robustez:')
print(f'  Media: {mean_acc:.2f}% | Desvio Padrao: {std_acc:.2f}%')
if std_acc < 5:
    print('  Modelo ROBUSTO a variacao de equipamentos!')
elif std_acc < 10:
    print('  Robustez moderada.')
else:
    print('  Modelo sensivel ao equipamento.')

In [ ]:
plt.figure(figsize=(12, 6))
acc_sorted = acc_by_machine.sort_values('accuracy', ascending=True)
colors_by_acc = [plt.cm.RdYlGn(acc/100) for acc in acc_sorted['accuracy']]
plt.barh(acc_sorted['machine'], acc_sorted['accuracy'], color=colors_by_acc)
plt.xlabel('Acuracia (%)')
plt.title('Acuracia por Equipamento de Raio-X (Test Set)')
plt.xlim(0, 105)
for i, (_, row) in enumerate(acc_sorted.iterrows()):
    plt.text(row['accuracy'] + 0.5, i,
             f'{row["accuracy"]:.1f}% (n={int(row["count"])})',
             va='center', fontsize=10)
plt.axvline(x=mean_acc, color='blue', linestyle='--', alpha=0.7, label=f'Media: {mean_acc:.1f}%')
plt.axvline(x=100*test_acc, color='red', linestyle=':', alpha=0.7, label=f'Global: {100*test_acc:.1f}%')
plt.legend()
plt.tight_layout()
plt.savefig('acuracia_por_equipamento.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nTabela Resumo:')
print(acc_by_machine.to_string(index=False))

---
## 9. Analise de Robustez Avancada: LOEO Cross-Validation

> **Leave-One-Equipment-Out (LOEO):** Treinamos o modelo em 6 equipamentos e
> testamos no 7o (held-out), repetindo para todos os 7. Isso mede a capacidade
> de generalizacao do modelo para equipamentos **nunca vistos** durante o
> treinamento — o teste mais rigoroso de robustez a domain shift.
>
> Diferente da avaliacao no split padrao (secao 8), onde o modelo ja viu
> todos os equipamentos durante o treino, o LOEO revela se o modelo consegue
> generalizar para dominios completamente invisiveis.

In [ ]:
# ── Prepara dados consolidados para LOEO ──────────────────────────────────────
# Combina todas as imagens (train+valid+test) em um unico DataFrame
df_all = pd.concat([df_train, df_valid, df_test], ignore_index=True)
print(f'Total de imagens para LOEO: {len(df_all)}')

# Adiciona coluna de equipamento
df_all['machine'] = df_all['ceph_id'].map(ceph_to_machine)

# Remove imagens sem mapeamento
df_all = df_all.dropna(subset=['machine']).reset_index(drop=True)
print(f'Apos remover sem mapeamento: {len(df_all)} imagens')

all_machines = sorted(df_all['machine'].unique())
print(f'Equipamentos: {all_machines}')
print(f'Distribuicao:')
for m in all_machines:
    count = (df_all['machine'] == m).sum()
    print(f'  {m:30s}: {count:4d} imagens')

In [ ]:
def run_loeo_fold(held_out_machine: str, df_all: pd.DataFrame,
                  num_epochs: int = 20, lr: float = 5e-5) -> dict:
    """Executa uma unica dobra LOEO."""
    # Divide dados
    df_train_fold = df_all[df_all['machine'] != held_out_machine].copy()
    df_test_fold  = df_all[df_all['machine'] == held_out_machine].copy()

    print(f'  Treino: {len(df_train_fold)} imagens ({df_train_fold["machine"].nunique()} equip.)')
    print(f'  Teste:  {len(df_test_fold)} imagens ({held_out_machine})')

    # Calcula pesos para balanceamento de classes no fold
    fold_counts = df_train_fold['label'].value_counts().reindex(CLASS_NAMES, fill_value=1)
    fold_weights = torch.tensor(
        [1.0 / max(fold_counts[c], 1) for c in CLASS_NAMES], dtype=torch.float
    ).to(DEVICE)
    fold_weights = fold_weights / fold_weights.sum() * NUM_CLASSES

    # Cria datasets e loaders (apenas imagens originais)
    train_ds_fold = CVMDataset(df_train_fold, transform=transform_train)
    test_ds_fold  = CVMDataset(df_test_fold,  transform=transform_eval)

    # Divide treino em train/val (80/20) para early stopping
    n_train = len(train_ds_fold)
    indices = list(range(n_train))
    random.shuffle(indices)
    split = int(0.8 * n_train)
    train_idx, val_idx = indices[:split], indices[split:]

    train_sub = Subset(train_ds_fold, train_idx)
    val_sub   = Subset(train_ds_fold, val_idx)

    train_loader_fold = DataLoader(train_sub, batch_size=BATCH_SIZE,
                                    shuffle=True, num_workers=0)
    val_loader_fold   = DataLoader(val_sub, batch_size=BATCH_SIZE,
                                    shuffle=False, num_workers=0)
    test_loader_fold  = DataLoader(test_ds_fold, batch_size=BATCH_SIZE,
                                    shuffle=False, num_workers=0)

    # Constroi modelo (fine-tuning completo para LOEO)
    model_fold = build_resnet50(NUM_CLASSES, freeze_strategy='full').to(DEVICE)
    criterion_fold = nn.CrossEntropyLoss(weight=fold_weights)
    optimizer_fold = optim.Adam(model_fold.parameters(), lr=lr)

    # Treinamento rapido com early stopping
    best_val = 0.0
    patience = 5
    patience_counter = 0

    for epoch in range(1, num_epochs + 1):
        train_loss, train_acc = train_one_epoch(
            model_fold, train_loader_fold, criterion_fold, optimizer_fold, DEVICE)
        val_loss, val_acc, _, _ = evaluate(
            model_fold, val_loader_fold, criterion_fold, DEVICE)

        if val_acc > best_val:
            best_val = val_acc
            patience_counter = 0
            # Salva melhor estado
            best_state = copy.deepcopy(model_fold.state_dict())
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'    Early stopping na epoca {epoch}')
                break

    # Carrega melhor modelo e avalia no held-out
    model_fold.load_state_dict(best_state)
    _, test_acc, preds, labels = evaluate(
        model_fold, test_loader_fold, criterion_fold, DEVICE)

    # Cohen Kappa
    kappa_fold = cohen_kappa_score(labels, preds, weights='quadratic')

    print(f'  >> Held-out: {held_out_machine} | Acc: {100*test_acc:.2f}% | Kappa: {kappa_fold:.4f}')

    return {
        'held_out': held_out_machine,
        'n_train': len(df_train_fold),
        'n_test': len(df_test_fold),
        'accuracy': test_acc,
        'kappa': kappa_fold,
        'model_state': best_state
    }

> **ATENCAO:** A celula abaixo executa 7 treinos completos (um para cada equipamento
> held-out). Cada treino usa ~20 epocas. Em CPU, pode levar varias horas.
> Em GPU (se disponivel), deve levar ~30-60 minutos.

In [ ]:
# ── Executa LOEO Cross-Validation ─────────────────────────────────────────────
print('='*70)
print('INICIANDO LEAVE-ONE-EQUIPMENT-OUT CROSS-VALIDATION')
print(f'Equipamentos: {len(all_machines)} | Epocas por fold: {LOEO_EPOCHS}')
print('='*70 + '\n')

loeo_results = []

for i, held_out in enumerate(all_machines):
    print(f'\n--- Fold {i+1}/{len(all_machines)}: Held-Out = {held_out} ---')
    result = run_loeo_fold(
        held_out, df_all, num_epochs=LOEO_EPOCHS, lr=LOEO_LR
    )
    loeo_results.append(result)

    # Salva modelo do fold
    fold_path = LOEO_DIR / f'model_loeo_{held_out.replace(" ", "_")}.pth'
    torch.save(result['model_state'], str(fold_path))
    print(f'  Modelo salvo em {fold_path}')

print('\n' + '='*70)
print('LOEO CROSS-VALIDATION CONCLUIDA!')
print('='*70)

In [ ]:
# ── Resultados Consolidados da LOEO ────────────────────────────────────────────
loeo_df = pd.DataFrame([{
    'Held-Out Machine': r['held_out'],
    'Train Size': r['n_train'],
    'Test Size': r['n_test'],
    'Accuracy (%)': round(100 * r['accuracy'], 2),
    'Cohen Kappa': round(r['kappa'], 4)
} for r in loeo_results])

print('\n' + '='*70)
print('RESULTADOS LOEO CROSS-VALIDATION')
print('='*70)
print(loeo_df.to_string(index=False))

# Estatisticas agregadas
mean_loeo_acc = loeo_df['Accuracy (%)'].mean()
std_loeo_acc  = loeo_df['Accuracy (%)'].std()
mean_loeo_kappa = loeo_df['Cohen Kappa'].mean()

print(f'\nResumo LOEO:')
print(f'  Acuracia Media (equipamentos nunca vistos): {mean_loeo_acc:.2f}%')
print(f'  Desvio Padrao: {std_loeo_acc:.2f}%')
print(f'  Cohen Kappa Medio: {mean_loeo_kappa:.4f}')
print(f'  Min: {loeo_df["Accuracy (%)"].min():.2f}% | Max: {loeo_df["Accuracy (%)"].max():.2f}%')

print(f'\nInterpretacao:')
if mean_loeo_acc > 50:
    print(f'  Acuracia > 50% em equipamentos nunca vistos: PROMISSOR!')
if std_loeo_acc < 8:
    print(f'  Baixa variacao entre equipamentos: ROBUSTO!')
if mean_loeo_acc > 50 and std_loeo_acc < 8:
    print(f'  A analise por equipamento mostra que o modelo generaliza bem!')

In [ ]:
# ── Grafico LOEO ───────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Grafico 1: Acuracia por held-out
loeo_sorted = loeo_df.sort_values('Accuracy (%)', ascending=True)
colors = [plt.cm.RdYlGn(acc/100) for acc in loeo_sorted['Accuracy (%)']]
axes[0].barh(loeo_sorted['Held-Out Machine'], loeo_sorted['Accuracy (%)'], color=colors)
axes[0].set_xlabel('Acuracia (%)')
axes[0].set_title('LOEO Cross-Validation: Acuracia por Equipamento Held-Out')
axes[0].set_xlim(0, 105)
for i, (_, row) in enumerate(loeo_sorted.iterrows()):
    axes[0].text(row['Accuracy (%)'] + 0.5, i,
                 f'{row["Accuracy (%)"]:.1f}% (n={int(row["Test Size"])})',
                 va='center', fontsize=9)
axes[0].axvline(x=mean_loeo_acc, color='blue', linestyle='--', alpha=0.7,
                label=f'Media: {mean_loeo_acc:.1f}%')
axes[0].legend()

# Grafico 2: Cohen Kappa por held-out
loeo_sorted_k = loeo_df.sort_values('Cohen Kappa', ascending=True)
colors_k = [plt.cm.RdYlGn(k/1.0) for k in loeo_sorted_k['Cohen Kappa']]
axes[1].barh(loeo_sorted_k['Held-Out Machine'], loeo_sorted_k['Cohen Kappa'], color=colors_k)
axes[1].set_xlabel('Cohen Kappa (quadratico)')
axes[1].set_title('LOEO: Concordancia (Cohen Kappa) por Equipamento Held-Out')
axes[1].set_xlim(0, 1.0)
for i, (_, row) in enumerate(loeo_sorted_k.iterrows()):
    axes[1].text(row['Cohen Kappa'] + 0.01, i,
                 f'{row["Cohen Kappa"]:.3f} (n={int(row["Test Size"])})',
                 va='center', fontsize=9)
axes[1].axvline(x=mean_loeo_kappa, color='blue', linestyle='--', alpha=0.7,
                label=f'Media: {mean_loeo_kappa:.3f}')
axes[1].legend()

plt.tight_layout()
plt.savefig(str(LOEO_DIR / 'loeo_results.png'), dpi=150, bbox_inches='tight')
plt.show()

# Salva resultados em CSV
loeo_df.to_csv(str(LOEO_DIR / 'loeo_results.csv'), index=False)
print(f'Resultados LOEO salvos em: {LOEO_DIR}/')

In [ ]:
# ── Comparacao: LOEO vs. Split Padrao ──────────────────────────────────────────
print('\n' + '='*70)
print('COMPARACAO: LOEO vs. SPLIT PADRAO')
print('='*70)
print()
print(f'  Split Padrao (Secao 7):                             {100*test_acc:.2f}%')
print(f'    - Modelo ve dados de TODOS os equipamentos no treino')
print(f'    - Testado no split pre-definido (70/15/15)')
print()
print(f'  LOEO Cross-Validation (Secao 9):                   {mean_loeo_acc:.2f}% +- {std_loeo_acc:.2f}%')
print(f'    - Modelo NUNCA ve o equipamento testado')
print(f'    - Mede generalizacao para dominios invisiveis')
print()

gap = 100*test_acc - mean_loeo_acc
print(f'  Diferenca (Split - LOEO): {gap:.2f}%')
if gap < 10:
    print(f'  Gap < 10%: O modelo generaliza bem para equipamentos invisiveis!')
elif gap < 20:
    print(f'  Gap entre 10-20%: Generalizacao moderada.')
else:
    print(f'  Gap > 20%: Modelo ainda dependente do equipamento de treino.')
    print(f'  Isso indica que o modelo capturou padroes especificos do equipamento')

---
## 10. Conclusoes e Proximos Passos

### Resumo da Metodologia

- **Classificacao Direta (Estagio Unico):** ResNet-50 com fine-tuning aplicado
  diretamente sobre as radiografias cefalometricas completas.
- **Pre-processamento:** CLAHE para padronizacao de contraste + aumento de dados
  (rotacao, crop, flip, ajustes de cor).
- **Avaliacao de Robustez:** Acuracia estratificada por cada um dos 7 equipamentos
  de raio-X no split padrao + LOEO Cross-Validation para medir generalizacao.

### Analise de Domain Shift

- A **acuracia por equipamento** revela em quais aparelhos o modelo tem
  melhor/pior desempenho, identificando vieses de dominio.
- O **desvio padrao entre equipamentos** quantifica a robustez geral: quanto
  menor, mais consistente o modelo entre diferentes fontes de aquisicao.
- O **LOEO** mede a capacidade de generalizar para equipamentos nunca vistos,
  sendo o indicador mais rigoroso de domain shift.

### Proximos Passos Sugeridos

1. **Testar com Vision Transformer (ViT)** como alternativa a ResNet-50.
2. **Adicionar Cross-Validation K-Fold** para avaliar a estabilidade do modelo
   independentemente da divisao treino/teste.
3. **Adicionar validacao externa** com outro dataset de radiografias cefalometricas.
4. **Analise de Grad-CAM** para verificar se o modelo realmente foca nas vertebras
   C2-C4 ou em artefatos de fundo.